In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

import os
import glob
import pandas as pd

# Set environment variables before importing PySpark
# Use the Java/Hadoop installation configured in your environment.
# Use the Java/Hadoop installation configured in your environment.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Initialize Spark session
spark = SparkSession.builder \
    .appName('TLC_FHVHV_Ingestion') \
    .master('local[*]') \
    .config('spark.driver.memory', '8g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.parquet.enableVectorizedReader', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version: {spark.version}')
print('Environment ready!')

# 01 — Data Ingestion: FHVHV Trip Records (2021–2025)

**Project:** NYC TLC Ridesharing Analysis Pipeline  
**Phase:** 1 — Data Foundation  
**Goal:** Load, inspect, and validate the raw FHVHV Parquet files for 2021–2025

**Local directories:** configure paths in the first code cell.
**Local directories:** configure paths in the first code cell.

---

### What this notebook does:
1. Set up PySpark environment
2. Load all 60 FHVHV Parquet files (Jan 2021 – Dec 2025)
3. Inspect schema and column types
4. Check data quality (nulls, duplicates, invalid values)
5. Summarize basic statistics
6. Save summary outputs for reference

## 0. Environment Setup

## 1. Define File Paths

In [ ]:
# Raw data path
RAW_DATA_PATH = str(RAW_DIR)

# Project paths
PROJECT_PATH = str(PROJECT_DIR)
PROCESSED_PATH = os.path.join(PROJECT_PATH, 'data', 'processed', 'combined')
OUTPUTS_PATH = os.path.join(PROJECT_PATH, 'outputs', 'tables')

# Target years
TARGET_YEARS = ['2021', '2022', '2023', '2024', '2025']

# Get all 2021-2025 parquet files
all_files = glob.glob(os.path.join(RAW_DATA_PATH, 'fhvhv_tripdata_*.parquet'))
target_files = sorted([
    f for f in all_files
    if any(year in os.path.basename(f) for year in TARGET_YEARS)
])

print(f'Total files found for 2021-2025: {len(target_files)}')
print('\nFiles to be loaded:')
for f in target_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'  {os.path.basename(f):45s} {size_mb:>8.1f} MB')

total_size = sum(os.path.getsize(f) for f in target_files) / (1024**3)
print(f'\nTotal size: {total_size:.2f} GB')

## 2. Load Data into PySpark

In [ ]:
print('Loading FHVHV parquet files...')

df = None
for file in target_files:
    temp_df = spark.read.parquet(file)

    if df is None:
        df = temp_df
    else:
        df = df.unionByName(temp_df,allowMissingColumns=True)
    print(f"loaded:{file} | Rows: {temp_df.count()}")


total_rows = df.count()
print(f'Data loaded successfully!')
print(f'Total rows:    {total_rows:,}')
print(f'Total columns: {len(df.columns)}')

## 3. Inspect Schema

In [ ]:
print('=== SCHEMA ===')
df.printSchema()

In [ ]:
print('=== SAMPLE ROWS ===')
df.show(5, truncate=False, vertical = True)

In [ ]:
print('=== COLUMN LIST ===')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')

## 4. Data Quality Check

In [ ]:
print('=== NULL VALUE COUNTS ===')

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_df = null_counts.toPandas().T.reset_index()
null_df.columns = ['column', 'null_count']
null_df['null_pct'] = (null_df['null_count'] / total_rows * 100).round(2)
null_df = null_df.sort_values('null_count', ascending=False)

print(null_df.to_string(index=False))

In [ ]:
print('=== DUPLICATE CHECK ===')
distinct = df.distinct().count()
duplicates = total_rows - distinct
print(f'Total rows:    {total_rows:,}')
print(f'Distinct rows: {distinct:,}')
print(f'Duplicates:    {duplicates:,} ({duplicates/total_rows*100:.2f}%)')

In [ ]:
print('=== DATE RANGE ===')
df.select(
    F.min('pickup_datetime').alias('earliest_trip'),
    F.max('pickup_datetime').alias('latest_trip')
).show(truncate=False)

In [ ]:
print('=== COMPANY BREAKDOWN ===')
print('HV0002 = Via, HV0003 = Uber, HV0005 = Lyft')
print()
df.groupBy('hvfhs_license_num') \
    .count() \
    .withColumn('pct', F.round(F.col('count') / total_rows * 100, 2)) \
    .orderBy(F.desc('count')) \
    .show()

## 5. Basic Statistics

In [ ]:
print('=== MONTHLY TRIP COUNTS ===')
monthly = df.withColumn('year_month', F.date_format('pickup_datetime', 'yyyy-MM')) \
    .groupBy('year_month') \
    .count() \
    .orderBy('year_month')

monthly.show(30)

In [ ]:
print('=== NUMERIC SUMMARY STATISTICS ===')
numeric_cols = ['trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'tips', 'congestion_charging', 'driver_pay']
existing_cols = [c for c in numeric_cols if c in df.columns]
df.select(existing_cols).describe().show()

In [ ]:
print('=== TOP 10 PICKUP ZONES ===')
df.groupBy('PULocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)

print('=== TOP 10 DROPOFF ZONES ===')
df.groupBy('DOLocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)

## 6. Congestion Pricing Pre/Post Split

NYC Congestion Pricing started **January 5, 2025** — our key policy cutoff.

In [ ]:
POLICY_DATE = '2025-01-05'

df_tagged = df.withColumn(
    'policy_period',
    F.when(F.col('pickup_datetime') < F.lit(POLICY_DATE), 'pre_policy')
     .otherwise('post_policy')
)

print('=== PRE vs POST CONGESTION PRICING ===')
df_tagged.groupBy('policy_period') \
    .count() \
    .withColumn('pct', F.round(F.col('count') / total_rows * 100, 2)) \
    .orderBy('policy_period') \
    .show()

In [ ]:
print('=== TOP 10 PICKUP ZONES === BEFORE ===')

df_tagged.filter(df_tagged['policy_period'] == 'pre_policy').groupBy('PULocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)

print('=== TOP 10 PICKUP ZONES === AFTER ===')
df_tagged.filter(df_tagged['policy_period'] == 'post_policy').groupBy('PULocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)

print('=== TOP 10 DROPOFF ZONES === BEFORE ===')
df_tagged.filter(df_tagged['policy_period'] == 'pre_policy').groupBy('DOLocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)
print('=== TOP 10 DROPOFF ZONES === AFTER ===')
df_tagged.filter(df_tagged['policy_period'] == 'post_policy').groupBy('DOLocationID') \
    .count() \
    .orderBy(F.desc('count')) \
    .show(10)

## 7. Save Summary Outputs

In [ ]:
# Save monthly counts
monthly_pd = monthly.toPandas()
monthly_file = os.path.join(OUTPUTS_PATH, 'fhvhv_monthly_counts_2021_2025.csv')
monthly_pd.to_csv(monthly_file, index=False)
print(f'Monthly counts saved to: {monthly_file}')

# Save null summary
null_file = os.path.join(OUTPUTS_PATH, 'fhvhv_null_summary.csv')
null_df.to_csv(null_file, index=False)
print(f'Null summary saved to:   {null_file}')

print('\nIngestion complete! Next step: 02_data_processing.ipynb')

---
## Notes

**Key columns for analysis:**
- `pickup_datetime` / `dropoff_datetime` — temporal analysis
- `PULocationID` / `DOLocationID` — spatial analysis  
- `hvfhs_license_num` — company identifier (Uber vs Lyft)
- `trip_miles` / `trip_time` — trip characteristics
- `base_passenger_fare` / `tips` / `tolls` — fare analysis

**Next step:** `02_data_processing.ipynb` — cleaning & feature engineering